# NOVA AI - GPU Server sur Google Colab

Transforme Colab en serveur IA avec GPU.
Ton PC se connecte à ce serveur via internet.

**Exécute les cellules dans l'ordre (▶️).**

## 1. Installation d'Ollama

In [ ]:
import time
print("Installation d'Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve &
time.sleep(3)
print("✓ Ollama prêt")

## 2. Téléchargement du modèle

Change `MODEL_NAME` si tu veux un autre modèle.

In [ ]:
MODEL_NAME = "qwen2.5-coder:7b"  # 7B = bon équilibre qualité/vitesse
print(f"Téléchargement de {MODEL_NAME}...")
!ollama pull $MODEL_NAME
print(f"✓ {MODEL_NAME} prêt")

## 3. Exposition du serveur

Cette cellule crée un tunnel internet. L'URL apparaîtra en bas.

In [ ]:
print("Installation du tunnel (localtunnel)...")
!npm install -g localtunnel 2>&1 | tail -1

import subprocess, time, re, requests

proc = subprocess.Popen(
    ['lt', '--port', '11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("Connexion...")

for _ in range(30):
    line = proc.stdout.readline()
    if line:
        txt = line.strip()
        if txt:
            print(txt)
        match = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', txt)
        if match:
            public_url = match.group()
            break
    time.sleep(0.5)

if not public_url:
    # Réessaie avec une autre approche
    import asyncio
    !npm install -g localtunnel 2>/dev/null
    print("\nNouvelle tentative...")
    proc2 = subprocess.Popen(
        ['lt', '--port', '11434', '--print-requests'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for _ in range(40):
        line = proc2.stdout.readline()
        if line:
            txt = line.strip()
            if txt:
                print(txt)
            match = re.search(r'https?://[^\s"]+', txt)
            if match:
                url = match.group().rstrip('.').rstrip(',')
                if 'loca.lt' in url or 'localtunnel' in url:
                    public_url = url
                    break
        time.sleep(0.5)
    proc = proc2

if public_url:
    print("\n" + "="*60)
    print("  ✅ SERVEUR PRÊT !")
    print(f"  🌐 URL : {public_url}")
    print("="*60)
    print("")
    print("  Copie cette URL dans NOVA AI : Settings → Remote Ollama URL")
    print("  Puis clique Test, puis Save.")
    # Test
    try:
        r = requests.get(f"{public_url}/api/tags", timeout=5)
        print(f"  ✅ Test connexion réussi ! Modèles: {r.json()}")
    except Exception as e:
        print(f"  ⚠️ Test: {e}")
else:
    print("\n⚠️ URL non trouvée. Vérifie que la cellule précédente a fini.")
    print("")
    print("Solution de secours :")
    print("1. Ouvre un NOUVEL onglet Colab")
    print("2. Exécute : !npm install -g localtunnel")
    print("3. Exécute : !lt --port 11434")
    print("4. L'URL s'affiche dans la sortie")

try:
    while True:
        time.sleep(30)
except:
    proc.terminate()
    print("Tunnel arrêté.")

---
## Si l'URL ne s'affiche pas

1. Réexécute la cellule du tunnel
2. Vérifie que la cellule 2 a bien fini de télécharger
3. Sinon, utilise NOVA en local (c'est déjà fonctionnel)